# SLC7A5 pos neg analysis

In [ ]:
  import gseapy as gp                                                                                                                       
                                                                                                                                            
  gene = 'SLC7A5'                                                                                                                           
                                                                                                                                            
  # ── 1. SLC7A5 pos/neg 레이블 부착 ──────────────────────────────                                                                         
  gene_idx = list(adata.var_names).index(gene)
  expr_vec = adata.X[:, gene_idx]                                                                                                           
  if hasattr(expr_vec, 'toarray'):
      expr_vec = expr_vec.toarray().flatten()                                                                                               
  else:                                                                                                                                     
      expr_vec = np.array(expr_vec).flatten()
                                                                                                                                            
  adata.obs['SLC7A5_group'] = np.where(expr_vec > 0, 'pos', 'neg')

  # ── 2. 3개 celltype 필터 ───────────────────────────────────────                                                                         
  # ※ Hepatocyte가 'celltype' 컬럼에 없으면 아래에서 확인
  # print(adata.obs['celltypist_liver'].unique())                                                                                           
  target_celltypes = ['Hepatocyte', 'Stellete', 'Macrophage','Endothelial','Lymphocyte']                                                                                  
  adata_sub = adata[adata.obs['celltype'].isin(target_celltypes)].copy()       

In [ ]:
from pathlib import Path
import re

# ── 3. 5개 celltype별 SLC7A5 pos vs neg DEG 저장 ───────────────
deg_outdir = Path("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/Nucseq/")
deg_outdir.mkdir(parents=True, exist_ok=True)

deg_results = {}
deg_tables = []

for ct in target_celltypes:
    sub = adata_sub[adata_sub.obs['celltype'] == ct].copy()
    grp_counts = sub.obs['SLC7A5_group'].value_counts()

    if 'pos' not in grp_counts or 'neg' not in grp_counts or grp_counts.min() < 3:
        print(f"[{ct}] pos/neg 세포 부족, skip: {grp_counts.to_dict()}")
        continue

    sc.tl.rank_genes_groups(
        sub,
        groupby='SLC7A5_group',
        groups=['pos'],
        reference='neg',
        method='wilcoxon',
        use_raw=False,
        pts=True
    )

    deg = sc.get.rank_genes_groups_df(sub, group='pos').rename(columns={'names': 'gene'})
    deg.insert(0, 'celltype', ct)
    deg.insert(1, 'comparison', 'SLC7A5_pos_vs_neg')
    deg.insert(2, 'n_pos', int(grp_counts['pos']))
    deg.insert(3, 'n_neg', int(grp_counts['neg']))

    safe_ct = re.sub(r'[^A-Za-z0-9_.-]+', '_', ct)
    full_path = deg_outdir / f"{safe_ct}_SLC7A5_pos_vs_neg_DEG_all.csv"
    sig_path = deg_outdir / f"{safe_ct}_SLC7A5_pos_vs_neg_DEG_padj0.05_logfc0.25.csv"

    deg.to_csv(full_path, index=False)
    sig = deg[(deg['pvals_adj'] < 0.05) & (deg['logfoldchanges'].abs() > 0.25)].copy()
    sig.to_csv(sig_path, index=False)

    deg_results[ct] = deg
    deg_tables.append(deg)
    print(f"[{ct}] saved: all={len(deg):,}, sig={len(sig):,} -> {full_path}")

if deg_tables:
    deg_all = pd.concat(deg_tables, ignore_index=True)
    combined_path = deg_outdir / "All_celltypes_SLC7A5_pos_vs_neg_DEG_all.csv"
    deg_all.to_csv(combined_path, index=False)
    print(f"Combined DEG saved: {combined_path} ({len(deg_all):,} rows)")

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

deg_outdir = Path("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/SLCposneg_nucseq/")
deg_outdir.mkdir(parents=True, exist_ok=True)

LFC_CAP = 10.0

deg_results = {}
deg_tables = []

for ct in target_celltypes:
    sub = adata_sub[adata_sub.obs['celltype'] == ct].copy()
    grp_counts = sub.obs['SLC7A5_group'].value_counts()

    if 'pos' not in grp_counts or 'neg' not in grp_counts or grp_counts.min() < 3:
        print(f"[{ct}] pos/neg 세포 부족, skip: {grp_counts.to_dict()}")
        continue

    sc.tl.rank_genes_groups(
        sub,
        groupby='SLC7A5_group',
        groups=['pos'],
        reference='neg',
        method='wilcoxon',
        use_raw=False,
        pts=True
    )

    deg = sc.get.rank_genes_groups_df(sub, group='pos').rename(
        columns={'names': 'gene'}
    )

    # ── 디버깅: 처음 한 번만 컬럼 확인 ─────────────────────
    if ct == target_celltypes[0]:
        print(f"[DEBUG] DEG columns: {deg.columns.tolist()}")

    # ── pct 컬럼명 자동 탐지 (scanpy 버전별 차이 대응) ──────
    pos_pct_col = next((c for c in deg.columns 
                        if c.startswith('pct_nz') and ('group' in c or c.endswith('_pos'))), None)
    neg_pct_col = next((c for c in deg.columns 
                        if c.startswith('pct_nz') and ('reference' in c or c.endswith('_neg'))), None)

    # 만약 pct_nz_reference가 없으면 adata에서 직접 계산
    if neg_pct_col is None:
        print(f"[{ct}] pct_nz_reference 컬럼 없음 -> 직접 계산")
        # neg 그룹의 cell mask
        neg_mask = (sub.obs['SLC7A5_group'] == 'neg').values
        X_neg = sub[neg_mask].X
        if hasattr(X_neg, 'toarray'):
            # sparse -> 발현 비율 계산
            pct_neg = np.asarray((X_neg > 0).mean(axis=0)).flatten()
        else:
            pct_neg = (X_neg > 0).mean(axis=0)
        gene_to_pct_neg = dict(zip(sub.var_names, pct_neg))
        deg['pct_nz_reference'] = deg['gene'].map(gene_to_pct_neg)
        neg_pct_col = 'pct_nz_reference'

    if pos_pct_col is None:
        pos_mask = (sub.obs['SLC7A5_group'] == 'pos').values
        X_pos = sub[pos_mask].X
        if hasattr(X_pos, 'toarray'):
            pct_pos = np.asarray((X_pos > 0).mean(axis=0)).flatten()
        else:
            pct_pos = (X_pos > 0).mean(axis=0)
        gene_to_pct_pos = dict(zip(sub.var_names, pct_pos))
        deg['pct_nz_group'] = deg['gene'].map(gene_to_pct_pos)
        pos_pct_col = 'pct_nz_group'

    # ── logFC NaN/inf 처리 ─────────────────────────────
    deg['logfoldchanges'] = pd.to_numeric(deg['logfoldchanges'], errors='coerce')
    deg['logFC_imputed'] = deg['logfoldchanges'].isna() | np.isinf(deg['logfoldchanges'])

    # vectorized 처리 (apply보다 빠르고 명확)
    lfc = deg['logfoldchanges'].values
    p = deg[pos_pct_col].values.astype(float)
    n = deg[neg_pct_col].values.astype(float)

    new_lfc = np.empty_like(lfc, dtype=float)

    for i in range(len(lfc)):
        v = lfc[i]
        if np.isfinite(v):
            new_lfc[i] = np.clip(v, -LFC_CAP, LFC_CAP)
        elif v == np.inf:
            new_lfc[i] = LFC_CAP
        elif v == -np.inf:
            new_lfc[i] = -LFC_CAP
        else:  # NaN
            pi, ni = p[i], n[i]
            if np.isnan(pi) and np.isnan(ni):
                new_lfc[i] = 0.0
            else:
                pi = 0.0 if np.isnan(pi) else pi
                ni = 0.0 if np.isnan(ni) else ni
                if pi > ni:
                    new_lfc[i] = LFC_CAP
                elif ni > pi:
                    new_lfc[i] = -LFC_CAP
                else:
                    new_lfc[i] = 0.0

    deg['logfoldchanges'] = new_lfc

    deg.insert(0, 'celltype', ct)
    deg.insert(1, 'comparison', 'SLC7A5_pos_vs_neg')
    deg.insert(2, 'n_pos', int(grp_counts['pos']))
    deg.insert(3, 'n_neg', int(grp_counts['neg']))

    safe_ct = re.sub(r'[^A-Za-z0-9_.-]+', '_', ct)
    full_path = deg_outdir / f"{safe_ct}_SLC7A5_pos_vs_neg_DEG_all.csv"
    sig_path = deg_outdir / f"{safe_ct}_SLC7A5_pos_vs_neg_DEG_padj0.05_logfc0.25.csv"

    deg.to_csv(full_path, index=False)
    sig = deg[(deg['pvals_adj'] < 0.05) & (deg['logfoldchanges'].abs() > 0.25)].copy()
    sig.to_csv(sig_path, index=False)

    deg_results[ct] = deg
    deg_tables.append(deg)
    print(f"[{ct}] saved: all={len(deg):,}, sig={len(sig):,}, "
          f"imputed_logFC={int(deg['logFC_imputed'].sum())} -> {full_path}")

if deg_tables:
    deg_all = pd.concat(deg_tables, ignore_index=True)
    combined_path = deg_outdir / "All_celltypes_SLC7A5_pos_vs_neg_DEG_all.csv"
    deg_all.to_csv(combined_path, index=False)
    print(f"Combined DEG saved: {combined_path} ({len(deg_all):,} rows)")

In [ ]:
  # ── 3. celltype별 GSEA ─────────────────────────────────────────                                                                         
  gene_sets = 'MSigDB_Hallmark_2020'   # 또는 로컬 .gmt 파일 경로
  results   = {}                                                                                                                            
                  
  for ct in target_celltypes:                                                                                                               
      sub = adata_sub[adata_sub.obs['celltype'] == ct].copy()

      grp_counts = sub.obs['SLC7A5_group'].value_counts()                                                                                   
      if 'pos' not in grp_counts or 'neg' not in grp_counts or grp_counts.min() < 3:
          print(f"[{ct}] pos/neg 세포 부족, skip")                                                                                          
          continue                                                                                                                          
                                                                                                                                            
      # pos vs neg Wilcoxon → ranked gene list                                                                                              
      sc.tl.rank_genes_groups(
          sub,                                                                                                                              
          groupby='SLC7A5_group',
          groups=['pos'],
          reference='neg',
          method='wilcoxon',
          use_raw=False
      )
      deg = sc.get.rank_genes_groups_df(sub, group='pos')
                                                                                                                                            
      ranked = (deg[['names', 'scores']]                                                                                                    
                .dropna()                                                                                                                   
                .drop_duplicates('names')                                                                                                   
                .sort_values('scores', ascending=False)
                .set_index('names')['scores'])

      # prerank GSEA                                                                                                                        
      pre_res = gp.prerank(
          rnk=ranked,                                                                                                                       
          gene_sets=gene_sets,
          threads=4,
          min_size=15,
          max_size=500,
          permutation_num=1000,
          outdir=None,                                                                                                                      
          seed=42,
          verbose=False                                                                                                                     
      )           

      results[ct] = pre_res
      print(f"[{ct}] 완료: {len(pre_res.res2d)} pathways")
                                                                                                                                            
  # ── 4. 결과 요약 출력 ─────────────────────────────────────────
  for ct, pre_res in results.items():                                                                                                       
      print(f"\n=== {ct} | SLC7A5 pos vs neg | top NES ===")
      print(pre_res.res2d.sort_values('NES', ascending=False)                                                                               
                         .head(10)[['Term', 'NES', 'NOM p-val', 'FDR q-val']]) 

In [ ]:
keywords = ['gly', 'fatty acid', 'lipid', 'cycle', 'hif', 'tor','pha']

for ct, pre_res in results.items():
    df = pre_res.res2d.copy()
    
    mask = df['Term'].str.lower().str.contains('|'.join(keywords))
    matched = df[mask].sort_values('NES', ascending=False)
    
    print(f"\n=== {ct} | keyword match ===")
    if matched.empty:
        print("No matching pathways found")
    else:
        print(matched[['Term', 'NES', 'NOM p-val', 'FDR q-val']])

In [ ]:
  # pos, neg 각각 모든 유전자 대비 GSEA
  results_sep = {}  # results_sep[ct]['pos'], results_sep[ct]['neg']                                                                        
                                                                                                                                            
  for ct in target_celltypes:                                                                                                               
      sub = adata_sub[adata_sub.obs['celltype'] == ct].copy()                                                                               
      results_sep[ct] = {}                                                                                                                  
   
      for grp, ref in [('pos', 'neg'), ('neg', 'pos')]:                                                                                     
          grp_cells = sub.obs['SLC7A5_group'] == grp
          if grp_cells.sum() < 3:                                                                                                           
              continue
                                                                                                                                            
          sc.tl.rank_genes_groups(
              sub,
              groupby='SLC7A5_group',                                                                                                       
              groups=[grp],
              reference=ref,                                                                                                                
              method='wilcoxon',
              use_raw=False
          )                                                                                                                                 
          deg = sc.get.rank_genes_groups_df(sub, group=grp)
          ranked = (deg[['names', 'scores']]                                                                                                
                    .dropna()                                                                                                               
                    .drop_duplicates('names')                                                                                               
                    .sort_values('scores', ascending=False)                                                                                 
                    .set_index('names')['scores'])                                                                                          
   
          pre_res = gp.prerank(                                                                                                             
              rnk=ranked,
              gene_sets=gene_sets,                                                                                                          
              threads=4,
              min_size=15,                                                                                                                  
              max_size=500,                                                                                                                 
              permutation_num=1000,
              outdir=None,                                                                                                                  
              seed=42,
              verbose=False                                                                                                                 
          )
          results_sep[ct][grp] = pre_res

In [ ]:
  # 관심 term 키워드 (대소문자 무관하게 매칭)                                                                                               
  keywords = [                                                                                                                              
      'MTOR', 'HIF1', 'AUTOPHAGY',
      'GLYCOLYSIS', 'GLUCONEOGENESIS', 'TCA', 'OXIDATIVE PHOSPHORYLATION',                                                                  
      'Fatty Acid'                                                                                                              
  ]                                                                                                                                         
                                                                                                                                            
  def match_terms(res_df, keywords):                                                                                                        
      matched = []
      for kw in keywords:                                                                                                                   
          hits = res_df[res_df['Term'].str.upper().str.contains(kw)]
          if not hits.empty:                                                                                                                
              matched.append(hits.iloc[0]['Term'])
      return matched     

In [ ]:
  # ── 1. 개별 enrichment plot ───────────────────────────────────                                                                          
  for ct, grp_dict in results_sep.items():                                                                                                  
      for grp, pre_res in grp_dict.items():
          matched = match_terms(pre_res.res2d, keywords)                                                                                    
          print(f"[{ct} | {grp}] matched: {matched}")
                                                                                                                                            
          for term in matched:
              row       = pre_res.res2d[pre_res.res2d['Term'] == term].iloc[0]
              term_data = pre_res.results.get(term, {})                                                                                     
              ranking   = term_data.get('RES', None)
              if ranking is None:                                                                                                           
                  continue
                                                                                                                                            
              hit_idx = (term_data.get('hit_indices')
                         or term_data.get('hits')
                         or term_data.get('hit_index')
                         or [])                                                                                                             
   
              color = '#3498DB' if grp == 'pos' else '#E74C3C'                                                                              
                  
              fig, ax = plt.subplots(figsize=(5, 3))                                                                                        
              ax.plot(ranking, color=color, lw=1.5)
              ax.axhline(0, color='black', lw=0.5, linestyle='--')                                                                          
              ax.axvline(np.argmax(np.abs(ranking)), color='red', lw=0.8, linestyle='--')
              for idx in hit_idx:                                                                                                           
                  ax.axvline(idx, color='grey', lw=0.3, alpha=0.4)
                                                                                                                                            
              nes = row['NES']
              fdr = row['FDR q-val']                                                                                                        
              ax.set_title(f"{ct} | {grp} | {term}\nNES={nes:.2f}, FDR={fdr:.3f}", fontsize=8)
              ax.set_xlabel('Rank', fontsize=7)                                                                                             
              ax.set_ylabel('Enrichment Score', fontsize=7)
              ax.tick_params(labelsize=6)                                                                                                   
              plt.tight_layout()
              #plt.savefig(f'GSEA_{ct}_{term[:30]}.pdf', bbox_inches='tight')
              plt.show()                                                                                                                  

In [ ]:
  def sig_star(p):                                                                                                                          
      if np.isnan(p): return ''                                                                                                             
      if p < 0.001:   return '***'                                                                                                          
      if p < 0.01:    return '**'                                                                                                           
      if p < 0.05:    return '*'
      return ''

  def fmt_pval(p):
      if np.isnan(p): return 'NA'
      return f'{p:.2e}' if p < 0.001 else f'{p:.3f}'

  # all_terms 수집 — ct → grp → pre_res 구조로                                                                                              
  all_terms = set()
  for ct, grp_dict in results_sep.items():                                                                                                  
      for grp, pre_res in grp_dict.items():
          all_terms.update(match_terms(pre_res.res2d, keywords))                                                                            
  all_terms = sorted(all_terms)
                                                                                                                                            
  celltypes_list = list(results_sep.keys())
  linestyles = ['-', '--', ':']
  colors     = {'pos': '#3498DB', 'neg': '#E74C3C'}                                                                                         
   
  n_pw = len(all_terms)                                                                                                                     
  fig, axes = plt.subplots(1, n_pw, figsize=(n_pw * 5, 4.5), constrained_layout=True)
  if n_pw == 1:                                                                                                                             
      axes = [axes]
                                                                                                                                            
  for ax, term in zip(axes, all_terms):
      legend_handles = []                                                                                                                   
                  
      for ci, ct in enumerate(celltypes_list):
          ls = linestyles[ci % len(linestyles)]
                                                                                                                                            
          for grp in ['pos', 'neg']:
              if grp not in results_sep[ct]:                                                                                                
                  continue
                                                                                                                                            
              pre_res   = results_sep[ct][grp]
              term_data = pre_res.results.get(term, {})                                                                                     
              es_curve  = term_data.get('RES', None)
              if es_curve is None or len(es_curve) == 0:
                  continue                                                                                                                  
   
              row = pre_res.res2d[pre_res.res2d['Term'] == term]                                                                            
              if row.empty:
                  continue

              nes  = row.iloc[0]['NES']                                                                                                     
              fdr  = row.iloc[0]['FDR q-val']
                                                                                                                                            
              color = colors[grp]
              x     = np.linspace(0, 1, len(es_curve))
              ax.plot(x, es_curve, color=color, lw=2.0, alpha=0.85, ls=ls)                                                                  
   
              star = sig_star(fdr)                                                                                                          
              lbl  = f"{ct} ({grp})  NES={nes:.2f}  FDR={fmt_pval(fdr)}{' '+star if star else ''}"
              legend_handles.append(                                                                                                        
                  plt.Line2D([0], [0], color=color, lw=2, ls=ls, label=lbl)
              )                                                                                                                             
                  
      ax.axhline(0, color='grey', lw=0.8, ls='--')                                                                                          
      ax.set_title(term, fontsize=10, fontweight='bold')
      ax.set_xlabel('Gene rank (normalized)', fontsize=8)                                                                                   
      ax.set_ylabel('Running ES', fontsize=8)
      ax.spines['top'].set_visible(False)                                                                                                   
      ax.spines['right'].set_visible(False)
      ax.legend(handles=legend_handles, fontsize=7, framealpha=0.35,                                                                        
                loc='best', title='line=celltype / color=group', title_fontsize=7)
                                                                                                                                            
  fig.suptitle(   
      'GSEA ES Curves — SLC7A5 pos (blue) vs neg (red) per celltype\n'                                                                      
      '(line style = celltype)',
      fontsize=12, fontweight='bold'                                                                                                        
  )
  #plt.savefig('GSEA_ES_curves_pos_neg.pdf', bbox_inches='tight', dpi=300)                                                                   
  plt.show()  

In [ ]:
  # index를 "Myeloid_pos", "Myeloid_neg" 형태로                                                                                             
  all_terms = set()                                                                                                                         
  for ct, grp_dict in results_sep.items():                                                                                                  
      for grp, pre_res in grp_dict.items():                                                                                                 
          all_terms.update(match_terms(pre_res.res2d, keywords))
  all_terms = sorted(all_terms)
                                                                                                                                            
  row_index = [f"{ct}_{grp}" for ct in results_sep for grp in ['pos', 'neg']
               if grp in results_sep[ct]]                                                                                                   
                  
  nes_mat = pd.DataFrame(index=row_index, columns=all_terms, dtype=float)
  fdr_mat = pd.DataFrame(index=row_index, columns=all_terms, dtype=float)
                                                                                                                                            
  for ct, grp_dict in results_sep.items():
      for grp, pre_res in grp_dict.items():                                                                                                 
          idx = f"{ct}_{grp}"
          for term in all_terms:                                                                                                            
              row = pre_res.res2d[pre_res.res2d['Term'] == term]
              if not row.empty:                                                                                                             
                  nes_mat.loc[idx, term] = row.iloc[0]['NES']
                  fdr_mat.loc[idx, term] = row.iloc[0]['FDR q-val']                                                                         
   
  nes_mat  = nes_mat.astype(float)                                                                                                          
  fdr_mat  = fdr_mat.astype(float)
  sig_size = -np.log10(fdr_mat.fillna(1) + 1e-10)                                                                                           
   
  # pos=파랑, neg=빨강 (행 이름 기준)                                                                                                       
  def row_color(idx_name, nes):
      grp = idx_name.split('_')[-1]                                                                                                         
      return '#3498DB' if grp == 'pos' else '#E74C3C'                                                                                       
                                                                                                                                            
  fig, ax = plt.subplots(figsize=(len(all_terms) * 0.9 + 1, len(row_index) * 0.8 + 1))                                                      
                  
  for i, idx in enumerate(nes_mat.index):                                                                                                   
      for j, term in enumerate(nes_mat.columns):
          nes = nes_mat.loc[idx, term]                                                                                                      
          sz  = max(sig_size.loc[idx, term] * 100, 5)
          fdr = fdr_mat.loc[idx, term]
          if np.isnan(nes):                                                                                                                 
              continue
          color = row_color(idx, nes)                                                                                                       
          ax.scatter(j, i, s=sz, c=color, alpha=0.8, edgecolors='k', linewidths=0.4)
          if not np.isnan(fdr) and fdr < 0.05:                                                                                              
              ax.text(j, i, '*', ha='center', va='center', fontsize=9,
                      fontweight='bold', color='white')                                                                                     
                                                                                                                                            
  ax.set_xticks(range(len(all_terms)))
  ax.set_xticklabels(all_terms, rotation=40, ha='right', fontsize=8)                                                                        
  ax.set_yticks(range(len(row_index)))
  ax.set_yticklabels(nes_mat.index, fontsize=9)                                                                                             
  ax.set_xlim(-0.5, len(all_terms) - 0.5)
  ax.set_ylim(-0.5, len(row_index) - 0.5)                                                                                                   
  ax.grid(True, linestyle='--', alpha=0.3)                                                                                                  
  ax.set_title('SLC7A5 pos (blue) vs neg (red) | GSEA NES\n(size=-log10 FDR, *=FDR<0.05)', fontsize=10)
                                                                                                                                            
  for label, color in [('pos group', '#3498DB'), ('neg group', '#E74C3C')]:                                                                 
      ax.scatter([], [], c=color, s=80, label=label, alpha=0.8)                                                                             
  ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1), fontsize=7)                                                                        
                  
  plt.tight_layout()                                                                                                                        
  #plt.savefig('GSEA_bubble_summary.pdf', bbox_inches='tight')
  plt.show()    

In [ ]:
  import scanpy as sc
  # KEGG용 results_sep 별도 생성
  gene_sets_kegg = 'KEGG_2021_Human'                                                                                                        
  results_sep_kegg = {}                                                                                                                     
   
  for ct in target_celltypes:                                                                                                               
      sub = adata_sub[adata_sub.obs['celltype'] == ct].copy()                                                                             
      results_sep_kegg[ct] = {}                                                                                                             
   
      for grp, ref in [('pos', 'neg'), ('neg', 'pos')]:                                                                                     
          if (sub.obs['SLC7A5_group'] == grp).sum() < 3:                                                                                  
              continue                                                                                                                      
          sc.tl.rank_genes_groups(sub, groupby='SLC7A5_group',                                                                            
                                  groups=[grp], reference=ref,                                                                              
                                  method='wilcoxon', use_raw=False)                                                                       
          deg = sc.get.rank_genes_groups_df(sub, group=grp)                                                                                 
          ranked = (deg[['names', 'scores']].dropna()                                                                                       
                    .drop_duplicates('names')                                                                                               
                    .sort_values('scores', ascending=False)                                                                                 
                    .set_index('names')['scores'])                                                                                          
          pre_res = gp.prerank(rnk=ranked, gene_sets=gene_sets_kegg,                                                                      
                               threads=4, min_size=15, max_size=500,                                                                        
                               permutation_num=1000, outdir=None,                                                                           
                               seed=42, verbose=False)                                                                                      
          results_sep_kegg[ct][grp] = pre_res    

In [ ]:
keywords = ['pi3k','mtor', 'hif', 'auto',
      'glycol', 'tca', 'oxidative',                                                                  
      'fatty' ]

for ct, grp_dict in results_sep_kegg.items():                                                                                             
      for grp, pre_res in grp_dict.items():                                                                                                 
          df = pre_res.res2d.copy()                                                                                                         
          mask = df['Term'].str.lower().str.contains('|'.join(keywords))                                                                    
          matched = df[mask].sort_values('NES', ascending=False)                                                                            
          print(f"\n=== {ct} | {grp} | keyword match ===")
          if matched.empty:                                                                                                                 
              print("No matching pathways found")
          else:                                                                                                                             
              print(matched[['Term', 'NES', 'NOM p-val', 'FDR q-val']])

In [ ]:
  # 관심 term 키워드 (대소문자 무관하게 매칭)                                                                                               
  keywords = ['pi3k','mtor', 'hif', 'auto',
      'glycol', 'tca', 'oxidative',                                                                  
      'fatty' ]                                                                                                                                      
                                                                                                                                            
  def match_terms(res_df, keywords):                                                                                                        
      matched = []
      for kw in keywords:                                                                                                                   
          hits = res_df[res_df['Term'].str.lower().str.contains(kw.lower())]
          if not hits.empty:                                                                                                                
              matched.append(hits.iloc[0]['Term'])
      return matched     

In [ ]:
  # ── 1. 개별 enrichment plot ───────────────────────────────────                                                                          
  for ct, grp_dict in results_sep_kegg.items():                                                                                                  
      for grp, pre_res in grp_dict.items():
          matched = match_terms(pre_res.res2d, keywords)                                                                                    
          print(f"[{ct} | {grp}] matched: {matched}")
                                                                                                                                            
          for term in matched:
              row       = pre_res.res2d[pre_res.res2d['Term'] == term].iloc[0]
              term_data = pre_res.results.get(term, {})                                                                                     
              ranking   = term_data.get('RES', None)
              if ranking is None:                                                                                                           
                  continue
                                                                                                                                            
              hit_idx = (term_data.get('hit_indices')
                         or term_data.get('hits')
                         or term_data.get('hit_index')
                         or [])                                                                                                             
   
              color = '#3498DB' if grp == 'pos' else '#E74C3C'                                                                              
                  
              fig, ax = plt.subplots(figsize=(5, 3))                                                                                        
              ax.plot(ranking, color=color, lw=1.5)
              ax.axhline(0, color='black', lw=0.5, linestyle='--')                                                                          
              ax.axvline(np.argmax(np.abs(ranking)), color='red', lw=0.8, linestyle='--')
              for idx in hit_idx:                                                                                                           
                  ax.axvline(idx, color='grey', lw=0.3, alpha=0.4)
                                                                                                                                            
              nes = row['NES']
              fdr = row['FDR q-val']                                                                                                        
              ax.set_title(f"{ct} | {grp} | {term}\nNES={nes:.2f}, FDR={fdr:.3f}", fontsize=8)
              ax.set_xlabel('Rank', fontsize=7)                                                                                             
              ax.set_ylabel('Enrichment Score', fontsize=7)
              ax.tick_params(labelsize=6)                                                                                                   
              plt.tight_layout()
              #plt.savefig(f'GSEA_{ct}_{term[:30]}.pdf', bbox_inches='tight')
              plt.show()                                                                                                                  

In [ ]:
  def sig_star(p):                                                                                                                          
      if np.isnan(p): return ''                                                                                                             
      if p < 0.001:   return '***'                                                                                                          
      if p < 0.01:    return '**'                                                                                                           
      if p < 0.05:    return '*'
      return ''

  def fmt_pval(p):
      if np.isnan(p): return 'NA'
      return f'{p:.2e}' if p < 0.001 else f'{p:.3f}'

  # all_terms 수집 — ct → grp → pre_res 구조로                                                                                              
  all_terms = set()
  for ct, grp_dict in results_sep_kegg.items():                                                                                                  
      for grp, pre_res in grp_dict.items():
          all_terms.update(match_terms(pre_res.res2d, keywords))                                                                            
  all_terms = sorted(all_terms)
                                                                                                                                            
  celltypes_list = list(results_sep_kegg.keys())
  linestyles = ['-', '--', ':']
  colors     = {'pos': '#3498DB', 'neg': '#E74C3C'}                                                                                         
   
  n_pw = len(all_terms)                                                                                                                     
  fig, axes = plt.subplots(1, n_pw, figsize=(n_pw * 5, 4.5), constrained_layout=True)
  if n_pw == 1:                                                                                                                             
      axes = [axes]
                                                                                                                                            
  for ax, term in zip(axes, all_terms):
      legend_handles = []                                                                                                                   
                  
      for ci, ct in enumerate(celltypes_list):
          ls = linestyles[ci % len(linestyles)]
                                                                                                                                            
          for grp in ['pos', 'neg']:
              if grp not in results_sep_kegg[ct]:                                                                                                
                  continue
                                                                                                                                            
              pre_res   = results_sep_kegg[ct][grp]
              term_data = pre_res.results.get(term, {})                                                                                     
              es_curve  = term_data.get('RES', None)
              if es_curve is None or len(es_curve) == 0:
                  continue                                                                                                                  
   
              row = pre_res.res2d[pre_res.res2d['Term'] == term]                                                                            
              if row.empty:
                  continue

              nes  = row.iloc[0]['NES']                                                                                                     
              fdr  = row.iloc[0]['FDR q-val']
                                                                                                                                            
              color = colors[grp]
              x     = np.linspace(0, 1, len(es_curve))
              ax.plot(x, es_curve, color=color, lw=2.0, alpha=0.85, ls=ls)                                                                  
   
              star = sig_star(fdr)                                                                                                          
              lbl  = f"{ct} ({grp})  NES={nes:.2f}  FDR={fmt_pval(fdr)}{' '+star if star else ''}"
              legend_handles.append(                                                                                                        
                  plt.Line2D([0], [0], color=color, lw=2, ls=ls, label=lbl)
              )                                                                                                                             
                  
      ax.axhline(0, color='grey', lw=0.8, ls='--')                                                                                          
      ax.set_title(term, fontsize=10, fontweight='bold')
      ax.set_xlabel('Gene rank (normalized)', fontsize=8)                                                                                   
      ax.set_ylabel('Running ES', fontsize=8)
      ax.spines['top'].set_visible(False)                                                                                                   
      ax.spines['right'].set_visible(False)
      ax.legend(handles=legend_handles, fontsize=7, framealpha=0.35,                                                                        
                loc='best', title='line=celltype / color=group', title_fontsize=7)
                                                                                                                                            
  fig.suptitle(   
      'GSEA ES Curves — SLC7A5 pos (blue) vs neg (red) per celltype\n'                                                                      
      '(line style = celltype)',
      fontsize=12, fontweight='bold'                                                                                                        
  )
  plt.savefig('/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure/Nuc_KEGGGSEA_ES_curves_pos_neg.pdf', bbox_inches='tight', dpi=300)                                                                   
  plt.show()  

In [ ]:
# ── Nucseq add-on: SLC7A5 Top/Bottom GSEA in Hepatocyte/Macrophage/Stellate ──
# SLC7A5 expression percentile groups are compared within each celltype.
# Note: this nucseq object uses the celltype label "Stellete".

import re
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import gseapy as gp
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

NUC_H5AD = "/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_DB_nuc_stage.h5ad"
adata_nuc_tb = sc.read(NUC_H5AD)

GENE = "SLC7A5"
CELLTYPE_COL = "celltype"
TARGET_CELLTYPES_NUC = ["Hepatocyte", "Macrophage", "Stellete"]
GENE_SET_COLLECTIONS_NUC = {
    "hallmark": "MSigDB_Hallmark_2020",
    "kegg": "KEGG_2021_Human",
}
LFC_CAP = 10.0
MIN_CELLS_PER_GROUP = 25
OUTDIR_ANA_NUC = Path("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/Nucseq/SLC7A5_topbottom_pct_hepatocyte_macrophage_stellate")
OUTDIR_FIG = Path("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure")
OUTDIR_ANA_NUC.mkdir(parents=True, exist_ok=True)
OUTDIR_FIG.mkdir(parents=True, exist_ok=True)

available_celltypes = set(adata_nuc_tb.obs[CELLTYPE_COL].astype(str).unique())
missing_ct = [ct for ct in TARGET_CELLTYPES_NUC if ct not in available_celltypes]
if missing_ct:
    raise ValueError(f"Missing celltype labels in adata_nuc_tb.obs['{CELLTYPE_COL}']: {missing_ct}")

if GENE not in adata_nuc_tb.var_names:
    raise ValueError(f"{GENE} not found in adata_nuc_tb.var_names")

adata_nuc_tb = adata_nuc_tb[adata_nuc_tb.obs[CELLTYPE_COL].isin(TARGET_CELLTYPES_NUC)].copy()
expr_vec = adata_nuc_tb[:, GENE].X
expr_vec = expr_vec.toarray().flatten() if hasattr(expr_vec, "toarray") else np.asarray(expr_vec).flatten()
adata_nuc_tb.obs["SLC7A5_expr"] = expr_vec

def adaptive_pct(n_cells):
    if n_cells >= 1000:
        return 10
    elif n_cells >= 500:
        return 15
    else:
        return 20

def safe_name(x):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(x))

def run_slc7a5_topbottom_gsea_nuc(adata_in, celltypes, gene_sets, label):
    results = {}
    summary_rows = []

    print(f"=== Nucseq {label.upper()} | SLC7A5 high vs low ===")
    print(f"{'CellType':<12} {'Total':>8} {'Pct':>5} {'N/group':>8} {'Detected':>9} {'Detected%':>10}")
    print("-" * 68)

    for ct in celltypes:
        sub = adata_in[adata_in.obs[CELLTYPE_COL] == ct].copy()
        n = len(sub)
        pct = adaptive_pct(n)
        n_g = int(n * pct / 100)
        detected = int((sub.obs["SLC7A5_expr"] > 0).sum())
        detected_pct = detected / n * 100 if n else np.nan
        print(f"{ct:<12} {n:>8} {pct:>4}% {n_g:>8} {detected:>9} {detected_pct:>9.2f}%")

        if n_g < MIN_CELLS_PER_GROUP:
            print(f"[{ct}] skip (n_per_group={n_g} < {MIN_CELLS_PER_GROUP})")
            continue

        expr = sub.obs["SLC7A5_expr"].to_numpy()
        sorted_idx = np.argsort(expr)
        grp = np.full(n, "middle", dtype=object)
        grp[sorted_idx[:n_g]] = "SLC7A5_low"
        grp[sorted_idx[-n_g:]] = "SLC7A5_high"
        sub.obs["SLC7A5_pct_group"] = grp

        sub_tb = sub[sub.obs["SLC7A5_pct_group"] != "middle"].copy()
        group_counts = sub_tb.obs["SLC7A5_pct_group"].value_counts().to_dict()

        sc.tl.rank_genes_groups(
            sub_tb,
            groupby="SLC7A5_pct_group",
            groups=["SLC7A5_high"],
            reference="SLC7A5_low",
            method="wilcoxon",
            use_raw=False,
            pts=True,
        )
        deg = sc.get.rank_genes_groups_df(sub_tb, group="SLC7A5_high").rename(columns={"names": "gene"})
        deg["logfoldchanges"] = pd.to_numeric(deg["logfoldchanges"], errors="coerce").clip(-LFC_CAP, LFC_CAP)
        deg.insert(0, "celltype", ct)
        deg.insert(1, "comparison", "SLC7A5_high_vs_low_topbottom_pct")
        deg.insert(2, "n_high", int(group_counts.get("SLC7A5_high", 0)))
        deg.insert(3, "n_low", int(group_counts.get("SLC7A5_low", 0)))
        deg.insert(4, "top_bottom_pct", pct)
        deg.to_csv(OUTDIR_ANA_NUC / f"DEG_{safe_name(ct)}_SLC7A5_high_vs_low_{label}.csv", index=False)

        ranked = (
            deg[["gene", "scores"]]
            .dropna()
            .drop_duplicates("gene")
            .sort_values("scores", ascending=False)
            .set_index("gene")["scores"]
        )
        dup_pct = ranked.duplicated().mean() * 100

        pre_res = gp.prerank(
            rnk=ranked,
            gene_sets=gene_sets,
            threads=4,
            min_size=15,
            max_size=500,
            permutation_num=1000,
            outdir=None,
            seed=42,
            verbose=False,
        )
        results[ct] = pre_res
        pre_res.res2d.to_csv(OUTDIR_ANA_NUC / f"GSEA_{safe_name(ct)}_{label}_topbottom.csv", index=False)

        n_sig = int((pre_res.res2d["FDR q-val"].astype(float) < 0.25).sum())
        summary_rows.append({
            "gene_set": label,
            "celltype": ct,
            "total_cells": n,
            "top_bottom_pct": pct,
            "n_per_group": n_g,
            "slc7a5_detected_cells": detected,
            "slc7a5_detected_pct": detected_pct,
            "duplicated_score_pct": dup_pct,
            "sig_pathway_fdr_lt_0.25": n_sig,
        })
        print(f"[{ct}] n/group={n_g} | dup_score={dup_pct:.1f}% | sig_pathway(FDR<0.25)={n_sig}")
        print(pre_res.res2d.sort_values("NES", ascending=False)
                           .head(5)[["Term", "NES", "NOM p-val", "FDR q-val"]].to_string(index=False))
        print()

    return results, pd.DataFrame(summary_rows)

results_nuc_topbottom = {}
summary_nuc_topbottom = []
for label, gene_sets in GENE_SET_COLLECTIONS_NUC.items():
    res, summary = run_slc7a5_topbottom_gsea_nuc(
        adata_nuc_tb,
        TARGET_CELLTYPES_NUC,
        gene_sets,
        label,
    )
    results_nuc_topbottom[label] = res
    summary_nuc_topbottom.append(summary)

summary_nuc_topbottom = (
    pd.concat(summary_nuc_topbottom, ignore_index=True)
    if summary_nuc_topbottom else pd.DataFrame()
)
summary_nuc_topbottom.to_csv(
    OUTDIR_ANA_NUC / "SLC7A5_topbottom_hepatocyte_macrophage_stellate_summary.csv",
    index=False,
)
summary_nuc_topbottom


# suppl.Figure1 - CD14 high CD16 low SLC7A5 high monocyte

In [ ]:
genes = ['CD14','FCGR3A']
n_genes = len(genes)

fig, axs = plt.subplots(n_genes, 1, figsize=(10, 3 * n_genes))

# genes가 1개일 때 대비
if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata,
        keys=gene,
        groupby='celltype',
        ax=axs[i],
        use_raw=False,
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)
    plt.setp(axs[i].get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure/Fig2C_violinplot_NFKB.pdf", bbox_inches='tight', dpi=300)

In [ ]:
sc.pl.umap(adata, color=['celltype'])

In [ ]:
sc.pl.umap(adata, color=['CD14'])

In [ ]:
sc.pl.umap(adata, color=['FCGR3A'])

In [ ]:
# CD14 high / CD16(FCGR3A) low / SLC7A5 high macrophage enrichment in MASLD
from scipy.stats import mannwhitneyu, fisher_exact
from pathlib import Path

outdir = Path('/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/Nucseq/SupplFigure1_CD14hi_CD16lo_SLC7A5hi')
figdir = Path('/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure')
outdir.mkdir(parents=True, exist_ok=True)
figdir.mkdir(parents=True, exist_ok=True)

def get_expr_vec(adata_obj, gene):
    if gene not in adata_obj.var_names:
        raise ValueError(f'{gene} is not in adata.var_names')
    x = adata_obj[:, gene].X
    if sp.issparse(x):
        x = x.toarray()
    return np.asarray(x).ravel()

def positive_q_threshold(x, q=0.75):
    x = np.asarray(x)
    xpos = x[x > 0]
    if len(xpos) >= 20:
        return float(np.quantile(xpos, q))
    return 0.0

def high_mask(x, threshold):
    return x >= threshold if threshold > 0 else x > 0

In [ ]:
# This nucseq object labels monocytes/macrophages as 'Macrophage'.
mono_mask = adata.obs['celltype'].astype(str).eq('Macrophage')
mono = adata[mono_mask].copy()
print(f'Macrophage nuclei: {mono.n_obs:,}')

cd14 = get_expr_vec(mono, 'CD14')
fcgr3a = get_expr_vec(mono, 'FCGR3A')
slc7a5 = get_expr_vec(mono, 'SLC7A5')

cd14_thr = positive_q_threshold(cd14, 0.75)
fcgr3a_thr = float(np.quantile(fcgr3a, 0.25))
slc7a5_thr = positive_q_threshold(slc7a5, 0.75)

mono.obs['CD14_high'] = high_mask(cd14, cd14_thr)
mono.obs['FCGR3A_low'] = fcgr3a <= fcgr3a_thr
mono.obs['SLC7A5_high'] = high_mask(slc7a5, slc7a5_thr)
mono.obs['CD14hi_CD16lo_SLC7A5hi'] = (
    mono.obs['CD14_high'] & mono.obs['FCGR3A_low'] & mono.obs['SLC7A5_high']
)

print({
    'CD14_high_threshold': cd14_thr,
    'FCGR3A_low_threshold': fcgr3a_thr,
    'SLC7A5_high_threshold': slc7a5_thr,
})
print(mono.obs['CD14hi_CD16lo_SLC7A5hi'].value_counts().rename('n_cells'))

In [ ]:
# MASLD = MASL + MASH; compare against Healthy and exclude end stage/Cirrhosis here.
disease_map = {'Healthy': 'Healthy', 'MASL': 'MASLD', 'MASH': 'MASLD'}
mono.obs['MASLD_group'] = mono.obs['Disease_status'].astype(str).map(disease_map)
plot_obs = mono.obs.dropna(subset=['MASLD_group']).copy()
plot_obs['is_triple_high'] = plot_obs['CD14hi_CD16lo_SLC7A5hi'].astype(bool)

sample_frac = (
    plot_obs.groupby(['sample', 'MASLD_group'], observed=True)
    .agg(
        n_macrophage=('is_triple_high', 'size'),
        n_triple_high=('is_triple_high', 'sum'),
    )
    .reset_index()
)
sample_frac['fraction_triple_high'] = sample_frac['n_triple_high'] / sample_frac['n_macrophage']
sample_frac = sample_frac[sample_frac['n_macrophage'] >= 10].copy()
sample_frac.to_csv(outdir / 'sample_fraction_CD14hi_CD16lo_SLC7A5hi_macrophage.csv', index=False)

cell_count = pd.crosstab(plot_obs['MASLD_group'], plot_obs['is_triple_high'])
cell_count.to_csv(outdir / 'cell_count_CD14hi_CD16lo_SLC7A5hi_macrophage.csv')

healthy_frac = sample_frac.loc[sample_frac['MASLD_group'] == 'Healthy', 'fraction_triple_high']
masld_frac = sample_frac.loc[sample_frac['MASLD_group'] == 'MASLD', 'fraction_triple_high']
if len(healthy_frac) >= 2 and len(masld_frac) >= 2:
    mw = mannwhitneyu(masld_frac, healthy_frac, alternative='greater')
    mw_p = mw.pvalue
else:
    mw_p = np.nan

fisher_table = cell_count.reindex(index=['MASLD', 'Healthy'], columns=[True, False], fill_value=0).values
oddsratio, fisher_p = fisher_exact(fisher_table, alternative='greater')

summary = pd.DataFrame({
    'comparison': ['MASLD_vs_Healthy'],
    'n_samples_Healthy': [int((sample_frac['MASLD_group'] == 'Healthy').sum())],
    'n_samples_MASLD': [int((sample_frac['MASLD_group'] == 'MASLD').sum())],
    'mean_fraction_Healthy': [healthy_frac.mean()],
    'mean_fraction_MASLD': [masld_frac.mean()],
    'mannwhitney_p_greater': [mw_p],
    'cell_level_fisher_oddsratio': [oddsratio],
    'cell_level_fisher_p_greater': [fisher_p],
})
summary.to_csv(outdir / 'stats_CD14hi_CD16lo_SLC7A5hi_macrophage_MASLD_vs_Healthy.csv', index=False)
display(summary)
display(sample_frac.sort_values(['MASLD_group', 'fraction_triple_high'], ascending=[True, False]).head())

fig, ax = plt.subplots(figsize=(4, 4))
sns.boxplot(
    data=sample_frac,
    x='MASLD_group',
    y='fraction_triple_high',
    order=['Healthy', 'MASLD'],
    showfliers=False,
    palette={'Healthy': '#4C78A8', 'MASLD': '#E45756'},
    ax=ax,
)
sns.stripplot(
    data=sample_frac,
    x='MASLD_group',
    y='fraction_triple_high',
    order=['Healthy', 'MASLD'],
    color='black',
    size=4,
    jitter=0.2,
    alpha=0.7,
    ax=ax,
)
ax.set_xlabel('')
ax.set_ylabel('Fraction among macrophages')
ax.set_title('CD14hi CD16lo SLC7A5hi macrophages')
ax.text(0.5, 1.03, f'Mann-Whitney p={mw_p:.3g}', ha='center', va='bottom', transform=ax.transAxes)
plt.tight_layout()
#plt.savefig(figdir / 'SupplFigure1_CD14hi_CD16lo_SLC7A5hi_macrophage_fraction_MASLD.pdf', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
sc.pl.umap(
    mono,
    color=['CD14hi_CD16lo_SLC7A5hi', 'Disease_status'],
    wspace=0.35,
    save=None,
)

In [ ]:
# Suppl.Figure1: nucseq validation of CD14hi / CD16lo / SLC7A5hi monocyte-like cells in MASLD
# Method 1: Healthy-derived expression cutoff gating
# Method 2: macrophage reclustering and cluster-level fraction comparison

from pathlib import Path
from scipy.stats import mannwhitneyu, fisher_exact, zscore

outdir = Path('/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/02_Analysis/Nucseq/SupplFigure1_CD14hi_CD16lo_SLC7A5hi')
figdir = Path('/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure')
outdir.mkdir(parents=True, exist_ok=True)
figdir.mkdir(parents=True, exist_ok=True)

disease_map = {'Healthy': 'Healthy', 'MASL': 'MASLD', 'MASH': 'MASLD'}

def get_expr_vec(adata_obj, gene):
    if gene not in adata_obj.var_names:
        raise ValueError(f'{gene} is not in adata.var_names')
    x = adata_obj[:, gene].X
    if sp.issparse(x):
        x = x.toarray()
    return np.asarray(x).ravel()

def positive_quantile_threshold(x, q=0.75, fallback=0.0):
    x = np.asarray(x)
    xpos = x[x > 0]
    if len(xpos) >= 20:
        return float(np.quantile(xpos, q))
    return float(fallback)

def high_mask(x, threshold):
    return x >= threshold if threshold > 0 else x > 0

def summarize_fraction_by_sample(obs, flag_col, prefix, min_cells=10):
    tmp = obs.dropna(subset=['MASLD_group']).copy()
    tmp[flag_col] = tmp[flag_col].astype(bool)

    sample_frac = (
        tmp.groupby(['sample', 'MASLD_group'], observed=True)
        .agg(
            n_macrophage=(flag_col, 'size'),
            n_positive=(flag_col, 'sum'),
        )
        .reset_index()
    )
    sample_frac['fraction_positive'] = sample_frac['n_positive'] / sample_frac['n_macrophage']
    sample_frac = sample_frac[sample_frac['n_macrophage'] >= min_cells].copy()

    healthy_frac = sample_frac.loc[sample_frac['MASLD_group'] == 'Healthy', 'fraction_positive']
    masld_frac = sample_frac.loc[sample_frac['MASLD_group'] == 'MASLD', 'fraction_positive']
    if len(healthy_frac) >= 2 and len(masld_frac) >= 2:
        mw_p = mannwhitneyu(masld_frac, healthy_frac, alternative='greater').pvalue
    else:
        mw_p = np.nan

    cell_count = pd.crosstab(tmp['MASLD_group'], tmp[flag_col])
    fisher_table = cell_count.reindex(index=['MASLD', 'Healthy'], columns=[True, False], fill_value=0).values
    oddsratio, fisher_p = fisher_exact(fisher_table, alternative='greater')

    summary = pd.DataFrame({
        'method': [prefix],
        'n_samples_Healthy': [int((sample_frac['MASLD_group'] == 'Healthy').sum())],
        'n_samples_MASLD': [int((sample_frac['MASLD_group'] == 'MASLD').sum())],
        'mean_fraction_Healthy': [healthy_frac.mean()],
        'mean_fraction_MASLD': [masld_frac.mean()],
        'mannwhitney_p_greater': [mw_p],
        'cell_level_fisher_oddsratio': [oddsratio],
        'cell_level_fisher_p_greater': [fisher_p],
    })

    sample_frac.to_csv(outdir / f'{prefix}_sample_fraction.csv', index=False)
    cell_count.to_csv(outdir / f'{prefix}_cell_count.csv')
    summary.to_csv(outdir / f'{prefix}_stats.csv', index=False)
    return sample_frac, summary, cell_count

def plot_sample_fraction(sample_frac, summary, title, filename):
    fig, ax = plt.subplots(figsize=(4, 4))
    sns.boxplot(
        data=sample_frac,
        x='MASLD_group',
        y='fraction_positive',
        order=['Healthy', 'MASLD'],
        showfliers=False,
        palette={'Healthy': '#4C78A8', 'MASLD': '#E45756'},
        ax=ax,
    )
    sns.stripplot(
        data=sample_frac,
        x='MASLD_group',
        y='fraction_positive',
        order=['Healthy', 'MASLD'],
        color='black',
        size=4,
        jitter=0.2,
        alpha=0.7,
        ax=ax,
    )
    p = summary['mannwhitney_p_greater'].iloc[0]
    ax.set_xlabel('')
    ax.set_ylabel('Fraction among macrophages')
    ax.set_title(title)
    ax.text(0.5, 1.03, f'Mann-Whitney p={p:.3g}', ha='center', va='bottom', transform=ax.transAxes)
    plt.tight_layout()
    plt.savefig(figdir / filename, bbox_inches='tight', dpi=300)
    plt.show()

In [ ]:
# Common macrophage object for both methods.
mono = adata[adata.obs['celltype'].astype(str).eq('Macrophage')].copy()
mono.obs['MASLD_group'] = mono.obs['Disease_status'].astype(str).map(disease_map)
print(f'Macrophage nuclei: {mono.n_obs:,}')
print(mono.obs['MASLD_group'].value_counts(dropna=False))

In [ ]:
# -----------------------------------------------------------------------------
# Method 1. Healthy-derived cutoff gating
# CD14 high and SLC7A5 high: 75th percentile among positive Healthy macrophages.
# CD16 low: 25th percentile of FCGR3A in Healthy macrophages.
# -----------------------------------------------------------------------------
cd14 = get_expr_vec(mono, 'CD14')
fcgr3a = get_expr_vec(mono, 'FCGR3A')
slc7a5 = get_expr_vec(mono, 'SLC7A5')
healthy_mask = mono.obs['MASLD_group'].eq('Healthy').values

cd14_thr = positive_quantile_threshold(cd14[healthy_mask], q=0.75)
fcgr3a_thr = float(np.quantile(fcgr3a[healthy_mask], 0.25))
slc7a5_thr = positive_quantile_threshold(slc7a5[healthy_mask], q=0.75)

mono.obs['cutoff_CD14_high'] = high_mask(cd14, cd14_thr)
mono.obs['cutoff_FCGR3A_low'] = fcgr3a <= fcgr3a_thr
mono.obs['cutoff_SLC7A5_high'] = high_mask(slc7a5, slc7a5_thr)
mono.obs['cutoff_CD14hi_CD16lo_SLC7A5hi'] = (
    mono.obs['cutoff_CD14_high']
    & mono.obs['cutoff_FCGR3A_low']
    & mono.obs['cutoff_SLC7A5_high']
)

cutoff_info = pd.DataFrame({
    'gene_or_marker': ['CD14', 'FCGR3A_CD16', 'SLC7A5'],
    'criterion': ['Healthy positive-cell 75th percentile', 'Healthy all-cell 25th percentile', 'Healthy positive-cell 75th percentile'],
    'threshold': [cd14_thr, fcgr3a_thr, slc7a5_thr],
})
cutoff_info.to_csv(outdir / 'method1_healthy_derived_cutoffs.csv', index=False)
display(cutoff_info)

m1_frac, m1_summary, m1_count = summarize_fraction_by_sample(
    mono.obs,
    flag_col='cutoff_CD14hi_CD16lo_SLC7A5hi',
    prefix='method1_healthy_cutoff_CD14hi_CD16lo_SLC7A5hi',
)
display(m1_summary)
plot_sample_fraction(
    m1_frac,
    m1_summary,
    'Healthy-cutoff CD14hi CD16lo SLC7A5hi',
    'SupplFigure1_method1_healthy_cutoff_CD14hi_CD16lo_SLC7A5hi.pdf',
)

# Sensitivity check for SLC7A5 high cutoff.
sensitivity_rows = []
for slc_q in [0.50, 0.60, 0.75, 0.90]:
    slc_thr = positive_quantile_threshold(slc7a5[healthy_mask], q=slc_q)
    flag = f'cutoff_sensitivity_SLC7A5q{int(slc_q * 100)}'
    mono.obs[flag] = mono.obs['cutoff_CD14_high'] & mono.obs['cutoff_FCGR3A_low'] & high_mask(slc7a5, slc_thr)
    frac, stat, _ = summarize_fraction_by_sample(mono.obs, flag_col=flag, prefix=f'method1_sensitivity_SLC7A5q{int(slc_q * 100)}')
    row = stat.iloc[0].to_dict()
    row['SLC7A5_quantile'] = slc_q
    row['SLC7A5_threshold'] = slc_thr
    sensitivity_rows.append(row)
sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(outdir / 'method1_SLC7A5_cutoff_sensitivity.csv', index=False)
display(sensitivity)

sc.pl.umap(
    mono,
    color=['cutoff_CD14hi_CD16lo_SLC7A5hi', 'Disease_status'],
    wspace=0.35,
    show=True,
)

In [ ]:
# -----------------------------------------------------------------------------
# Method 2. Reclustering macrophages and identifying the CD14hi/CD16lo/SLC7A5hi-like cluster
# Cluster score = z(mean CD14) + z(mean SLC7A5) - z(mean FCGR3A).
# The top-scoring cluster is used as the nucseq cluster-level counterpart.
# -----------------------------------------------------------------------------
mono_clust = mono.copy()
sc.pp.highly_variable_genes(mono_clust, n_top_genes=2000, flavor='seurat')
mono_hvg = mono_clust[:, mono_clust.var['highly_variable']].copy()
sc.pp.scale(mono_hvg, max_value=10)
sc.tl.pca(mono_hvg, n_comps=30, svd_solver='arpack')
sc.pp.neighbors(mono_hvg, n_neighbors=15, n_pcs=30)
sc.tl.umap(mono_hvg)

In [ ]:
sc.tl.leiden(mono_hvg, resolution=0.5, key_added='leiden_macrophage')
sc.pl.umap(
    mono_hvg,
    color=['leiden_macrophage','CD14','SLC7A5']
)

In [ ]:
mono.obs['leiden_macrophage'] = mono_hvg.obs['leiden_macrophage'].astype(str).values
mono.obsm['X_umap_macrophage_recluster'] = mono_hvg.obsm['X_umap'].copy()

cluster_expr = []
for cl in sorted(mono.obs['leiden_macrophage'].unique(), key=lambda x: int(x) if str(x).isdigit() else str(x)):
    mask = mono.obs['leiden_macrophage'].eq(cl).values
    cluster_expr.append({
        'cluster': cl,
        'n_cells': int(mask.sum()),
        'mean_CD14': float(cd14[mask].mean()),
        'mean_FCGR3A': float(fcgr3a[mask].mean()),
        'mean_SLC7A5': float(slc7a5[mask].mean()),
        'pct_CD14_pos': float((cd14[mask] > 0).mean()),
        'pct_FCGR3A_pos': float((fcgr3a[mask] > 0).mean()),
        'pct_SLC7A5_pos': float((slc7a5[mask] > 0).mean()),
    })
cluster_expr = pd.DataFrame(cluster_expr)
for col in ['mean_CD14', 'mean_FCGR3A', 'mean_SLC7A5']:
    cluster_expr[f'z_{col}'] = zscore(cluster_expr[col].values, nan_policy='omit')
cluster_expr['CD14hi_CD16lo_SLC7A5hi_score'] = (
    cluster_expr['z_mean_CD14'] + cluster_expr['z_mean_SLC7A5'] - cluster_expr['z_mean_FCGR3A']
)
cluster_expr = cluster_expr.sort_values('CD14hi_CD16lo_SLC7A5hi_score', ascending=False).reset_index(drop=True)
top_cluster = cluster_expr.loc[0, 'cluster']
cluster_expr['selected_as_counterpart'] = cluster_expr['cluster'].eq(top_cluster)
cluster_expr.to_csv(outdir / 'method2_macrophage_recluster_marker_summary.csv', index=False)
display(cluster_expr)
print(f'Selected CD14hi/CD16lo/SLC7A5hi-like nucseq cluster: {top_cluster}')

mono.obs['cluster_CD14hi_CD16lo_SLC7A5hi_like'] = mono.obs['leiden_macrophage'].eq(top_cluster)
m2_frac, m2_summary, m2_count = summarize_fraction_by_sample(
    mono.obs,
    flag_col='cluster_CD14hi_CD16lo_SLC7A5hi_like',
    prefix='method2_recluster_CD14hi_CD16lo_SLC7A5hi_like_cluster',
)
display(m2_summary)
plot_sample_fraction(
    m2_frac,
    m2_summary,
    f'Recluster cluster {top_cluster}: CD14hi CD16lo SLC7A5hi-like',
    'SupplFigure1_method2_recluster_CD14hi_CD16lo_SLC7A5hi_like_cluster.pdf',
)

fig, axs = plt.subplots(1, 4, figsize=(16, 4))
sc.pl.embedding(mono, basis='umap_macrophage_recluster', color='leiden_macrophage', ax=axs[0], show=False, title='Macrophage clusters')
sc.pl.embedding(mono, basis='umap_macrophage_recluster', color='CD14', ax=axs[1], show=False, title='CD14')
sc.pl.embedding(mono, basis='umap_macrophage_recluster', color='FCGR3A', ax=axs[2], show=False, title='FCGR3A/CD16')
sc.pl.embedding(mono, basis='umap_macrophage_recluster', color='SLC7A5', ax=axs[3], show=False, title='SLC7A5')
plt.tight_layout()
#plt.savefig(figdir / 'SupplFigure1_method2_macrophage_recluster_markers.pdf', bbox_inches='tight', dpi=300)
plt.show()

combined_summary = pd.concat([m1_summary, m2_summary], ignore_index=True)
combined_summary.to_csv(outdir / 'combined_method1_method2_stats.csv', index=False)
display(combined_summary)


In [ ]:
mono_hvg

In [ ]:
sc.pl.umap(
    mono_hvg,
    color=['leiden_macrophage','CD14','SLC7A5']
)

# Nucseq validation: special monocyte score analysis

In [ ]:
sc.pl.umap(adata, color=['celltype'])

In [ ]:
adata_mac = adata[adata.obs["celltype"].isin(["Macrophage"])].copy()

In [ ]:
adata_mac

In [ ]:
# -----------------------------------------------------------------------------
# Recompute neighbors, Leiden clustering, and UMAP
# -----------------------------------------------------------------------------
if "X_umap" in adata_mac.obsm and "X_umap_original" not in adata_re.obsm:
    adata_mac.obsm["X_umap_original"] = adata_re.obsm["X_umap"].copy()

sc.pp.neighbors(
    adata_mac,
    n_neighbors=20,
    n_pcs=None if use_rep.startswith("X_scVI") else N_PCS,
    use_rep="X_scVI",
    key_added="mac_neighbors",
    random_state=SEED,
)

sc.tl.leiden(
    adata_mac,
    resolution=LEIDEN_RESOLUTION,
    key_added=CLUSTER_KEY,
    neighbors_key=NEIGHBORS_KEY,
    random_state=SEED,
)

sc.tl.umap(adata_mac, neighbors_key=NEIGHBORS_KEY, random_state=SEED)
adata_mac.obsm[UMAP_KEY] = adata_re.obsm["X_umap"].copy()

if "X_umap_original" in adata_mac.obsm:
    adata_re.obsm["X_umap"] = adata_mac.obsm["X_umap_original"].copy()

print(adata_mac.obs[CLUSTER_KEY].value_counts().sort_index())
